In [48]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================ CELL A0 : setup ================================
# audit.ipynb - manual audit of refutation cases (read-only, no API needed)
import os, json, glob, sys

sys.path.insert(0, "/content/drive/MyDrive/medical_qa_hallucination/experiments")
from config import DATASET_FINAL, RESULTS_DIR, RESULTS_FINAL, EXPERIMENTS

AUDIT_DIR = os.path.join(EXPERIMENTS, "audit")
os.makedirs(AUDIT_DIR, exist_ok=True)
AUDIT_CSV = os.path.join(AUDIT_DIR, "audit.csv")

GROUPS = ["reviewed20", "batch1", "batch2", "batch3", "batch4"]
ARM_CODES = {"coarse": "c", "atomic": "a",
             "atomic_gl_filter": "ag", "atomic_sec_filter": "as"}

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

DATASET = load_jsonl(DATASET_FINAL)
assert len(DATASET) == 160, f"dataset: {len(DATASET)} rows != 160"
DS_BY_UID = {r["uid"]: r for r in DATASET}

def _refs_str(uid):
    refs = DS_BY_UID.get(uid, {}).get("guideline_ref", "")
    if isinstance(refs, str):
        return refs
    parts = []
    for r in refs:
        if isinstance(r, dict):
            loc = (r.get("section") or r.get("table_id") or r.get("figure_id")
                   or r.get("box_id") or r.get("locator") or "")
            parts.append(str(loc))
        else:
            parts.append(str(r))
    return "; ".join(p for p in parts if p)

def _pick_run_file(folder, run_set, r):
    """All candidates for one run number; prefer a payload with no failures."""
    cands = sorted(glob.glob(os.path.join(folder, f"results_{run_set}_run{r}*.json")))
    assert cands, f"{run_set}_run{r}: no file found in {folder}"
    loaded = [(p, json.load(open(p, encoding="utf-8"))) for p in cands]
    good = [x for x in loaded if not x[1].get("failed")]
    pick = (good or loaded)[0]
    if len(loaded) > 1:
        print(f"  note: {run_set}_run{r} has {len(loaded)} files, "
              f"using {os.path.basename(pick[0])}")
    return pick

RUNS, RUN_FILES, UID2GROUP = {}, {}, {}
for g in GROUPS:
    folder = RESULTS_DIR if g == "reviewed20" else RESULTS_FINAL
    triple, paths = [], []
    for r in (1, 2, 3):
        p, d = _pick_run_file(folder, g, r)
        triple.append(d); paths.append(p)
    uid_sets = {tuple(sorted(d["uids"])) for d in triple}
    assert len(uid_sets) == 1, f"{g}: the 3 runs do not share the same uid set"
    RUNS[g], RUN_FILES[g] = triple, paths
    for u in triple[0]["uids"]:
        assert u not in UID2GROUP, f"{u} in both {UID2GROUP.get(u)} and {g}"
        UID2GROUP[u] = g
    print(f"{g:10s}: {len(triple[0]['uids'])} questions | "
          + ", ".join(os.path.basename(p) for p in paths))

ARM_ORDER = RUNS["reviewed20"][0]["arms"]
assert len(UID2GROUP) == 160, f"uids across groups: {len(UID2GROUP)} != 160"
assert set(UID2GROUP) == set(DS_BY_UID), "run uids do not match the dataset"
print("registry OK: 160 questions, 5 groups, 15 run files")
print("references sample check:", repr(_refs_str(RUNS["batch1"][0]["uids"][0])))

reviewed20: 20 questions | results_reviewed20_run1.json, results_reviewed20_run2.json, results_reviewed20_run3.json
batch1    : 35 questions | results_batch1_run1_20260918_081423.json, results_batch1_run2_20260918_085220.json, results_batch1_run3_20260918_093127.json
batch2    : 35 questions | results_batch2_run1_20260918_101819.json, results_batch2_run2_20260918_110845.json, results_batch2_run3_20260918_115710.json
batch3    : 35 questions | results_batch3_run1_20260918_125742.json, results_batch3_run2_20260918_135110.json, results_batch3_run3_20260918_145047.json
batch4    : 35 questions | results_batch4_run1_20260918_155356.json, results_batch4_run2_20260918_164730.json, results_batch4_run3_20260918_173241.json
registry OK: 160 questions, 5 groups, 15 run files
references sample check: '4.4'


In [ ]:
# ================= CELL A1 : summary tables + combined audit.csv =============
import pandas as pd

pd.set_option("display.max_colwidth", None)   # never truncate references
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

def ref_overview(g):
    """Triage table of refutations across the 3 runs of one group.
    Only questions with >=1 first-pass refutation in any run/arm are listed.
    ref columns        : arm codes with ref_init>0; '4' = all four arms; '0' = none
    unresolved columns : '<arm><r1>/<r2>/<r3>' per non-converged arm
                         (rN = refuted claims at iteration N); '0' = converged"""
    triple = RUNS[g]
    per_run = [{r["uid"]: r for r in d["results"]} for d in triple]
    rows_out = []
    for u in triple[0]["uids"]:
        if not any(u in pr and any(pr[u]["arms"][a]["ref_init"] > 0
                                   for a in ARM_ORDER) for pr in per_run):
            continue
        rec = {"uid": u}
        for i, pr in enumerate(per_run, 1):
            q = pr.get(u)
            ref_arms = [ARM_CODES[a] for a in ARM_ORDER if q["arms"][a]["ref_init"] > 0]
            rec[f"run{i}_ref"] = "4" if len(ref_arms) == 4 else (", ".join(ref_arms) or "0")
            unr = []
            for a in ARM_ORDER:
                arm = q["arms"][a]
                if arm["final_refuted"] > 0:
                    per_it = "/".join(str(h["refuted"]) for h in arm["history"])
                    unr.append(f"{ARM_CODES[a]}{per_it}")
            rec[f"run{i}_unresolved"] = ", ".join(unr) if unr else "0"
        rec["references"] = _refs_str(u)
        rows_out.append(rec)

    cols = (["uid"] + [f"run{i}_{k}" for i in (1, 2, 3)
                       for k in ("ref", "unresolved")] + ["references"])
    df = pd.DataFrame(rows_out, columns=cols)
    print("=" * 110)
    print(f"{g}: {len(df)} of {len(triple[0]['uids'])} questions with refutations")
    display(df)
    print()
    return df

def _uid_rank():
    order = pd.read_csv(AUDIT_CSV, dtype=str)["uid"].tolist()
    return {u: i for i, u in enumerate(order)}

def _sort_claims(df):
    rank = _uid_rank()
    df = df.copy()
    df["_o"] = df["uid"].map(lambda u: rank.get(u, len(rank)))
    return (df.sort_values(["_o", "claim_id"])
              .drop(columns="_o")
              .reset_index(drop=True))

ALL_OVERVIEW, total_flagged = {}, 0
for g in GROUPS:
    ALL_OVERVIEW[g] = ref_overview(g)
    total_flagged += len(ALL_OVERVIEW[g])
print("=" * 110)
print(f"TOTAL: {total_flagged} of 160 questions had >=1 first-pass refutation "
      f"in any run/arm")

# ---- combined audit csv (one file, all groups; preserves entered values) ---
frames = []
for g in GROUPS:
    d = ALL_OVERVIEW[g].copy()
    d.insert(1, "group", g)
    frames.append(d)
audit_df = pd.concat(frames, ignore_index=True)

if os.path.exists(AUDIT_CSV):
    old = pd.read_csv(AUDIT_CSV, dtype=str).fillna("")
    old = old.set_index("uid")[["audit_status", "audit_note"]]
    audit_df = audit_df.join(old, on="uid")
else:
    audit_df["audit_status"] = ""
    audit_df["audit_note"] = ""
audit_df["audit_status"] = audit_df["audit_status"].fillna("")
audit_df["audit_note"] = audit_df["audit_note"].fillna("")
audit_df.to_csv(AUDIT_CSV, index=False, encoding="utf-8")
print(f"\nSAVED combined audit csv -> {AUDIT_CSV}  ({len(audit_df)} uids)")
display(audit_df.iloc[:, :-2])

reviewed20: 13 of 20 questions with refutations


,uid,run1_ref,run1_unresolved,run2_ref,run2_unresolved,run3_ref,run3_unresolved,references
0,VHD_S7_Q2,"a, ag",0,0,0,0,0,7.4; Recommendation_Table_3; Figure_5
1,VHD_S8_Q1,0,0,0,0,as,0,8.2.1; Figure_6
2,VHD_S8_Q3,4,0,4,0,0,0,8.2.1; Figure_6
3,VHD_S8_Q9,4,a1/1/1,4,ag1/2/1,4,ag2/1/1,8.2.1; 8.4.2; Recommendation_Table_4
4,VHD_S9_Q1,0,0,0,0,4,0,9.2.4; Recommendation_Table_7; Table_7; Figure_13
5,VHD_S11_Q5,0,0,ag,0,0,0,11.2; Recommendation_Table_9
6,VHD_S12_Q1,4,0,4,0,0,0,12.2; 12.4; Recommendation_Table_10
7,AF_S3_Q1,4,a1/1/2,4,0,4,c1/1/2,Table_5; Recommendation_Table_1
8,AF_S6_Q1,0,0,ag,0,0,0,6.2.1; Recommendation_Table_7
9,AF_S6_Q2,4,as2/1/2,4,0,4,0,6.7.1; Recommendation_Table_12; Figure_10



batch1: 15 of 35 questions with refutations


,uid,run1_ref,run1_unresolved,run2_ref,run2_unresolved,run3_ref,run3_unresolved,references
0,VHD_S5_Q4,4,0,0,0,0,0,5.5.1
1,VHD_S6_Q6,"c, a, as",0,as,0,0,0,6.4
2,VHD_S14_Q1,"c, as",0,0,0,0,0,14.4.4; Figure_20
3,VHD_S14_Q3,0,0,"a, ag",0,a,0,14.4.3; 14.2
4,AF_S4_Q1,a,0,"c, a, ag",0,a,0,Figure_3
5,AF_S5_Q4,0,0,c,0,0,0,5.7; Recommendation_Table_5
6,AF_S6_Q6,c,0,0,0,"c, a, as",0,6.3; Recommendation_Table_8
7,AF_S6_Q8,"c, ag, as",0,0,0,0,0,6.2.1; Recommendation_Table_7
8,AF_S7_Q10,"a, as",0,"a, as",0,"c, a, as",0,7.1.1; Recommendation_Table_14
9,AF_S9_Q5,"a, ag",0,"a, ag",ag1/1/2,"a, ag",0,9.2; RecommendationTable_24; Figure_14



batch2: 21 of 35 questions with refutations


,uid,run1_ref,run1_unresolved,run2_ref,run2_unresolved,run3_ref,run3_unresolved,references
0,VHD_S6_Q4,as,0,"c, a, as",c1/2/1,4,0,Recommendation_Table_2; 6.2
1,VHD_S6_Q7,4,0,4,ag1/1/1,4,ag1/1/1,6.5
2,VHD_S8_Q8,4,0,4,as1/1/1,"c, ag",0,8.2.1; 8.4.1; Figure_6; Recommendation_Table_4
3,VHD_S8_Q10,4,0,0,0,0,0,8.3
4,VHD_S9_Q2,4,"c1/1/1, ag1/1/1",4,a1/1/1,4,ag1/1/1,Figure_10; 9.2.2
5,VHD_S9_Q12,0,0,as,0,0,0,9.2.4
6,VHD_S10_Q3,c,0,0,0,0,0,10.2.3; Table_8; Recommendation_Table_8
7,VHD_S13_Q1,0,0,as,0,"c, ag",0,13.3.1
8,AF_S3_Q4,4,0,4,0,4,0,3.6
9,AF_S6_Q5,c,0,a,0,0,0,6.1; Recommendation_Table_6


... (2 outputs left out)


batch4: 19 of 35 questions with refutations


,uid,run1_ref,run1_unresolved,run2_ref,run2_unresolved,run3_ref,run3_unresolved,references
0,VHD_S6_Q1,4,0,4,0,4,0,6.1; Recommendation_Table_1
1,VHD_S6_Q5,4,0,4,0,4,0,6.3
2,VHD_S7_Q7,as,0,as,0,as,0,7.3
3,VHD_S8_Q4,"a, ag, as",0,4,"a2/1/1, ag2/1/1",4,as1/1/2,8.4.1; Recommendation_Table_4
4,VHD_S8_Q7,"c, a, ag",0,4,ag2/1/1,0,0,8.2.1; Figure_6; 8.4.1; Recommendation_Table_4
5,VHD_S9_Q5,4,0,"c, a, ag",a1/1/1,"a, as",0,9.1.4; Recommendation_Table_6
6,VHD_S9_Q6,0,0,0,0,as,0,9.2.2; Figure_10
7,VHD_S9_Q8,as,0,0,0,0,0,9.1.2; Figure_10
8,VHD_S11_Q6,as,0,0,0,0,0,11.2; 11.4.2; Recommendation_Table_9; Figure_17; Figure_15
9,VHD_S14_Q8,0,0,0,0,as,0,14.3.2; Recommendation_Table_16; Figure_19



TOTAL: 91 of 160 questions had >=1 first-pass refutation in any run/arm

SAVED combined audit csv -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit.csv  (91 uids)


,uid,group,run1_ref,run1_unresolved,run2_ref,run2_unresolved,run3_ref,run3_unresolved,references
0,VHD_S7_Q2,reviewed20,"a, ag",0,0,0,0,0,7.4; Recommendation_Table_3; Figure_5
1,VHD_S8_Q1,reviewed20,0,0,0,0,as,0,8.2.1; Figure_6
2,VHD_S8_Q3,reviewed20,4,0,4,0,0,0,8.2.1; Figure_6
3,VHD_S8_Q9,reviewed20,4,a1/1/1,4,ag1/2/1,4,ag2/1/1,8.2.1; 8.4.2; Recommendation_Table_4
4,VHD_S9_Q1,reviewed20,0,0,0,0,4,0,9.2.4; Recommendation_Table_7; Table_7; Figure_13
...,...,...,...,...,...,...,...,...,...
86,AF_S9_Q10,batch4,4,0,"c, a, ag",0,0,0,9.7; Recommendation_Table_27
87,AF_S10_Q2,batch4,"ag, as",0,0,0,"ag, as",0,10.5.1; Recommendation_Table_32
88,CM_S7-4_Q1,batch4,4,0,"c, ag, as",0,0,0,7.4.4; Recommendation_Table_28
89,CVD_S3-3_Q1,batch4,4,ag2/1/2,4,a2/1/2,4,ag3/1/1,3.3.2


In [49]:
# ================= CELL A2 : interactive per-uid audit =======================
def audit_uid(uid):
    """All refutation details for ONE uid across all 3 runs of its group:
    first-pass refuted claims (the detection) and unresolved final-iteration
    claims (what survived the rewrite loop), plus the shared initial answer."""
    g = UID2GROUP.get(uid)
    if g is None:
        print(f"uid not found in any run set: {uid!r}")
        return
    print("#" * 96)
    print(f"UID {uid}  |  group: {g}  |  type: {DS_BY_UID[uid].get('question_type')}")
    print(f"DATASET guideline_ref: {_refs_str(uid)}")
    for i, d in enumerate(RUNS[g], 1):
        q = next((r for r in d["results"] if r["uid"] == uid), None)
        print("=" * 96)
        print(f"RUN {i}   file: {os.path.basename(RUN_FILES[g][i-1])}")
        if q is None:
            print("  [!] uid missing in this run")
            continue
        if not any(q["arms"][a]["ref_init"] > 0 for a in ARM_ORDER):
            print("  no refutations in this run")
            continue
        print("\nQUESTION:")
        print("  " + q["question"].replace("\n", "\n  "))
        print("\nINITIAL ANSWER (shared across arms, pre-verification):")
        print("  " + q["init_answer"].replace("\n", "\n  ") + "\n")
        for a in ARM_ORDER:
            arm = q["arms"][a]
            traj = "/".join(str(h["refuted"]) for h in arm["history"])
            if arm["ref_init"] == 0:
                print(f"  [{a}] ref_init: 0  (iters {arm['iterations']}, "
                      f"refuted/iter {traj})")
                continue
            print(f"  [{a}] ref_init: {arm['ref_init']}  (iters {arm['iterations']}, "
                  f"refuted/iter {traj}, final_refuted {arm['final_refuted']})")
            for v in arm["history"][0]["verified"]:
                if v["verdict"] == "refuted":
                    print(f"      REFUTED (iter 1): {v['claim']}")
                    print(f"      WHY             : {v['rationale']}")
                    print(f"      TARGET          : class {v.get('target_class')} "
                          f"| level {v.get('target_level')}")
                    print(f"      MODEL chunk_id  : {v.get('chunk_id')}")
                    print(f"      RETRIEVED ids   : {v.get('retrieved_ids')}")
            if arm["final_refuted"] > 0:
                print(f"      UNRESOLVED at final iteration:")
                for v in arm["history"][-1]["verified"]:
                    if v["verdict"] == "refuted":
                        print(f"        - {v['claim']}")
                        print(f"          why: {v['rationale']}")
            print(f"      FINAL ANSWER [{a}]:")
            print("        " + arm["final_answer"].replace("\n", "\n        "))
            print()

STATUS_OPTIONS = ["true_hallucination", "verifier_error", "cross_source",
                  "version_mismatch", "mixed", "other"]
STATUS_MEANINGS = {
    "true_hallucination": "claim contradicts the ESC guideline content it should come from",
    "verifier_error":     "claim was actually correct; retrieval/verifier misjudged (false positive)",
    "cross_source":       "claim is defensible from another source, just not in this ESC text",
    "version_mismatch":   "claim matches a different/older guideline version",
    "mixed":              "this uid's refuted claims fall into different categories (explain in note)",
    "other":              "anything else (explain in note)",
}

def audit_prompt(uid):
    """Prompt for audit_status + audit_note for one uid and save immediately.
    Empty input keeps the current value, so re-auditing is safe."""
    df = pd.read_csv(AUDIT_CSV, dtype=str).fillna("")
    if uid not in set(df["uid"]):
        print(f"{uid}: no refutations in any run (not in the audit set)")
        return
    row = df.index[df["uid"] == uid][0]
    if df.at[row, "audit_status"] or df.at[row, "audit_note"]:
        print(f"current entry: status={df.at[row,'audit_status']!r} "
              f"note={df.at[row,'audit_note']!r}")
    print("status options:")
    for i, s in enumerate(STATUS_OPTIONS, 1):
        print(f"  {i}. {s:20s} - {STATUS_MEANINGS[s]}")
    s_in = input("status (1-6 or name, empty = keep current): ").strip().lower()
    if s_in:
        if s_in.isdigit() and 1 <= int(s_in) <= len(STATUS_OPTIONS):
            df.at[row, "audit_status"] = STATUS_OPTIONS[int(s_in) - 1]
        elif s_in in STATUS_OPTIONS:
            df.at[row, "audit_status"] = s_in
        else:
            print(f"  invalid status {s_in!r}; keeping current")
    n_in = input("note (empty = keep current): ").strip()
    if n_in:
        df.at[row, "audit_note"] = n_in
    df.to_csv(AUDIT_CSV, index=False, encoding="utf-8")
    print(f"saved: {uid} -> status={df.at[row,'audit_status']!r}\n")

# interactive loop: print details, then label
uid = input("uid: ").strip()

if uid:
    audit_uid(uid)
    audit_prompt(uid)
else:
    print("No UID entered.")

KeyboardInterrupt: Interrupted by user

In [ ]:
print("..")

..


In [ ]:
upsert_claim("VHD_S8_Q4", "B", claim_text="Severe AS is strongly supported by CT aortic valve calcium scores greater than 3000 Agatston units in men or greater than 1600 AU in women",
    label="verifier_error", n_instances="8/12", arms_flagged="c, a, ag, as", runs_flagged="2, 3",
    cited_chunk="coarse:0103 / atomic:0163 (8.2.1) in all 8", source_chunk_retrieved="yes",
    outcome="inverted after 8.2.1 paste: 6 of 8 finals narrowed to >2000/>1200, dropping the corpus-consistent upper tier (correction-introduced removal of true corpus content); run2 a and run3 as shipped '>3000 highly likely' unresolved at final iteration, and that shipped content was itself corpus-consistent",
    note="Within-chunk evidence-selection phantom. 8.2.1 is two-tier: >2000 men / >1200 women indicate severe AS with high sensitivity and specificity (~85%); higher thresholds >3000 men / >1600 women are very specific; unlikely <1600/<800. The claim paraphrases the upper tier and is corpus-supported. The verifier quoted only the lower-tier sentence in all 8 instances and refuted the upper tier, ignoring it within the same chunk. Relabeled from the provisional true_hallucination after the 8.2.1 paste. Unflagged and more consequential: the Class I misattribution (Recommendation_Table_4 gives IIa B for preserved-EF LF-LG; all 3 initial answers claimed Class I; never refuted in any of the 12 instances, only run3 ag self-corrected). Also unflagged: run3 as final shipped 'mean gradient >=40 mmHg when severe obstruction is demonstrated', contradicting the <40 low-gradient definition.")

saved VHD_S8_Q4 B -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv


In [ ]:
# =============== CELL A3 : claim-level audit (audit_claims.csv) ==============
# Ground truth lives at claim level: one row per unique refuted claim TYPE
# per uid. uid-level audit.csv stays as the roll-up ("mixed" when rows differ).
CLAIMS_CSV = os.path.join(AUDIT_DIR, "audit_claims.csv")

CLAIM_FIELDS = ["uid", "claim_id", "claim_text", "label", "th_subtype",
                "n_instances", "arms_flagged", "runs_flagged", "cited_chunk",
                "source_chunk_retrieved", "outcome", "note"]

# labels: true_hallucination / verifier_error / cross_source /
#         version_mismatch / other
# source_chunk_retrieved: was the text containing the true value in top-5?
#   yes / no / partial (explain in note); n/a for true_hallucination
CLAIM_SEED = [
    dict(uid="VHD_S7_Q2", claim_id="A",
         claim_text="Aortic size alone strongly mandates aortic intervention "
                    "in the setting of planned valve surgery",
         label="verifier_error", n_instances="2/12", arms_flagged="a, ag",
         runs_flagged="1", cited_chunk="atomic:0163->0139 (prose 7.4)",
         source_chunk_retrieved="yes",
         outcome="converged; final answers kept correct 45 mm content",
         note="Claim true in vignette context (54 mm > 45 mm with AVR "
              "indicated; prose 7.4 verbatim). Decomposition lost numeric "
              "context; verifier over-read 'strongly mandates'. Retrieval "
              "pulled wrong Table_4 (AS) rows but cited correct prose chunk."),
    dict(uid="VHD_S8_Q1", claim_id="A",
         claim_text="AS calcium thresholds >1200 AU women / >2000 AU men",
         label="verifier_error", n_instances="1/12", arms_flagged="as",
         runs_flagged="3", cited_chunk="atomic:0163 (prose 8.2.1)",
         source_chunk_retrieved="yes (extraction-displaced)",
         outcome="converged; final answer has correct thresholds",
         note="Claim correct per original PDF and Figure_6. Corpus chunk has "
              "PDF-extraction displacement of number-sex pairs; verifier read "
              "it linearly and inverted the mapping. Figure_6 with correct "
              "explicit mapping was in top-5 but not used as evidence."),
    dict(uid="VHD_S8_Q3", claim_id="A",
         claim_text="Severe AS very likely when calcium score >3000 AU in men",
         label="verifier_error", n_instances="8/12", arms_flagged="4",
         runs_flagged="1, 2",
         cited_chunk="atomic:0163 / coarse:0103 (prose 8.2.1)",
         source_chunk_retrieved="yes",
         outcome="converged; final answers state >2000 AU (correct, tiered "
                 "detail lost)",
         note="Claim matches guideline 'very specific' tier (>3000 men), a "
              "subset of >2000; answer's tiered phrasing mirrored guideline. "
              "Verifier applied rigid single-threshold matching."),
    dict(uid="VHD_S8_Q3", claim_id="B",
         claim_text="Severe AS unlikely when calcium score <1600 AU in men",
         label="verifier_error", n_instances="1/12", arms_flagged="as",
         runs_flagged="2", cited_chunk="atomic:0190 (Figure_6)",
         source_chunk_retrieved="no (8.2.1 prose 0163 absent from top-5)",
         outcome="converged",
         note="Near-verbatim guideline ('severe AS becomes unlikely ... <1600 "
              "AU in men'). Verifier misread 'unlikely' as 'excluded'; "
              "retrieval miss, inferred from absence in Figure_6."),
    dict(uid="VHD_S8_Q9", claim_id="A",
         claim_text="Very severe AS defined by mean gradient >=60 mmHg",
         label="verifier_error", n_instances="10/12", arms_flagged="4",
         th_subtype="strength_logic",
         runs_flagged="1, 2, 3",
         cited_chunk="atomic:0163 / coarse:0103 (prose 8.2.1)",
         source_chunk_retrieved="no (definition is in Recommendation_Table_4 "
              "IIa asymptomatic row; coarse: Table_4 not retrieved; atomic: "
              "rows 0173/0175 retrieved, check if very-severe bullet attached)",
         outcome="unresolved at final in r1 atomic and r2 gl; final answers "
                 "correctly kept >=60 mmHg",
         note="Claim verbatim in Recommendation_Table_4 and in the question. "
              "Verifier anchored on 8.2.1 prose (no such threshold) and on "
              "the symptomatic row (matches its stated target class I B)."),
    dict(uid="VHD_S8_Q9", claim_id="B",
         claim_text="True AS severity best represented by maximal Doppler "
                    "value from any acoustic window",
         label="true_hallucination", n_instances="6/12", arms_flagged="4",
         runs_flagged="2, 3",
         cited_chunk="atomic:0163 / coarse:0103 (prose 8.2.1)",
         source_chunk_retrieved="yes",
         outcome="converged; rewrites restored multiparametric framing",
         note="Overstatement: guideline states mean pressure gradient is the "
              "most robust parameter and grading is multiparametric. Claim "
              "conflates window-selection technique with parameter hierarchy. "
              "Pipeline working as intended."),
]

def _load_claims():
    if os.path.exists(CLAIMS_CSV):
        return pd.read_csv(CLAIMS_CSV, dtype=str).fillna("")
    return pd.DataFrame(columns=CLAIM_FIELDS)

def upsert_claim(uid, claim_id, **fields):
    """Add or fully update one claim row, save immediately (like audit_prompt).
    Example: upsert_claim('VHD_S9_Q1', 'A', claim_text='...',
             label='verifier_error', n_instances='3/12', ...)"""
    df = _load_claims()
    row = {f: "" for f in CLAIM_FIELDS}
    if len(df):
        m = (df["uid"] == uid) & (df["claim_id"] == claim_id)
        if m.any():
            row.update(df[m].iloc[0].to_dict())
            df = df[~m]
    row.update(uid=uid, claim_id=claim_id, **fields)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)[CLAIM_FIELDS]
    df = _sort_claims(df)[CLAIM_FIELDS]
    df.to_csv(CLAIMS_CSV, index=False, encoding="utf-8")
    print(f"saved {uid} {claim_id} -> {CLAIMS_CSV}")

# apply seeds: refresh factual fields, preserve manually entered label/note
df = _load_claims()
preserved = 0
for seed in CLAIM_SEED:
    m = (df["uid"] == seed["uid"]) & (df["claim_id"] == seed["claim_id"])
    if m.any():
        for f in ("label", "note"):
            v = df[m].iloc[0][f]
            if v:
                seed[f] = v
                preserved += 1
        df = df[~m]
    df = pd.concat([df, pd.DataFrame([seed])], ignore_index=True)
df = _sort_claims(df)[CLAIM_FIELDS]
df.to_csv(CLAIMS_CSV, index=False, encoding="utf-8")
print(f"SAVED claim audit csv -> {CLAIMS_CSV}  "
      f"({len(df)} claim rows, {df['uid'].nunique()} uids, "
      f"{preserved} manual entries preserved)")

# ---- reconciliation: seed instance counts vs ref_init sums from run files ---
def _ref_init_total(uid):
    tot = 0
    for d in RUNS[UID2GROUP[uid]]:
        r = next(x for x in d["results"] if x["uid"] == uid)
        tot += sum(r["arms"][a]["ref_init"] for a in ARM_ORDER)
    return tot

ok = True
for uid, grp in df.groupby("uid", sort=False):
    seeded = sum(int(s.split("/")[0]) for s in grp["n_instances"])
    actual = _ref_init_total(uid)
    flag = "OK" if seeded == actual else "MISMATCH"
    ok &= seeded == actual
    print(f"  {uid}: claims rows sum {seeded} vs ref_init total {actual}  {flag}")
print("reconciliation", "passed" if ok else "FAILED: fix n_instances values")

SAVED claim audit csv -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv  (160 claim rows, 91 uids, 12 manual entries preserved)
  VHD_S7_Q2: claims rows sum 2 vs ref_init total 2  OK
  VHD_S8_Q1: claims rows sum 1 vs ref_init total 1  OK
  VHD_S8_Q3: claims rows sum 9 vs ref_init total 9  OK
  VHD_S8_Q9: claims rows sum 16 vs ref_init total 16  OK
  VHD_S9_Q1: claims rows sum 4 vs ref_init total 4  OK
  VHD_S11_Q5: claims rows sum 1 vs ref_init total 1  OK
  VHD_S12_Q1: claims rows sum 8 vs ref_init total 8  OK
  AF_S3_Q1: claims rows sum 12 vs ref_init total 12  OK
  AF_S6_Q1: claims rows sum 1 vs ref_init total 1  OK
  AF_S6_Q2: claims rows sum 26 vs ref_init total 26  OK
  AF_S6_Q3: claims rows sum 1 vs ref_init total 1  OK
  CM_S7-1_Q4: claims rows sum 1 vs ref_init total 1  OK
  CM_S7-2_Q1: claims rows sum 2 vs ref_init total 2  OK
  VHD_S5_Q4: claims rows sum 5 vs ref_init total 5  OK
  VHD_S6_Q6: claims rows sum 4 vs ref_init total 4  OK
  VHD

In [ ]:
display(df[df['uid']=="VHD_S8_Q4"][["uid", "claim_id", "label", "note"]])

,uid,claim_id,label,note
131,VHD_S8_Q4,A,true_hallucination,"Boundary-only conflict: Recommendation_Table_4 defines low flow as SVi <=35 mL/m², so normal flow is strictly >35; the claim assigns the boundary value to normal flow. True hallucination by the corpus-conflict rule but trivial in magnitude, a pedantic >=/> catch with clean convergence. Companion noise: run2 ag and run3 as logged self-defeating final-iteration refutations of the already corpus-correct '>35' wording (rationale restates the claim as correct), shipped unresolved; these sit outside ref_init but document final-iteration verifier incoherence."
132,VHD_S8_Q4,B,verifier_error,"Within-chunk evidence-selection phantom. 8.2.1 is two-tier: >2000 men / >1200 women indicate severe AS with high sensitivity and specificity (~85%); higher thresholds >3000 men / >1600 women are very specific; unlikely <1600/<800. The claim paraphrases the upper tier and is corpus-supported. The verifier quoted only the lower-tier sentence in all 8 instances and refuted the upper tier, ignoring it within the same chunk. Relabeled from the provisional true_hallucination after the 8.2.1 paste. Unflagged and more consequential: the Class I misattribution (Recommendation_Table_4 gives IIa B for preserved-EF LF-LG; all 3 initial answers claimed Class I; never refuted in any of the 12 instances, only run3 ag self-corrected). Also unflagged: run3 as final shipped 'mean gradient >=40 mmHg when severe obstruction is demonstrated', contradicting the <40 low-gradient definition."


In [ ]:
display(df[df["uid"] == "VHD_S8_Q9"][["uid", "claim_id", "label", "note"]])

,uid,claim_id,label,note
4,VHD_S8_Q9,A,verifier_error,Claim verbatim in Recommendation_Table_4 and in the question. Verifier anchored on 8.2.1 prose (no such threshold) and on the symptomatic row (matches its stated target class I B).
5,VHD_S8_Q9,B,true_hallucination,Overstatement: guideline states mean pressure gradient is the most robust parameter and grading is multiparametric. Claim conflates window-selection technique with parameter hierarchy. Pipeline working as intended.


In [ ]:
d = pd.read_csv(CLAIMS_CSV, dtype=str).fillna("")
d["n"] = d["n_instances"].str.split("/").str[0].astype(int)
print(d.groupby("label").agg(rows=("uid", "size"),
                             instances=("n", "sum"),
                             uids=("uid", "nunique")))

                    rows  instances  uids
label                                    
cross_source          18         50    16
true_hallucination    59        284    39
verifier_error        75        235    58
version_mismatch       8         37     6


In [ ]:
display(audit_df.iloc[:, [0, -2, -1]])

,uid,audit_status,audit_note
0,VHD_S7_Q2,verifier_error,"Claim true in context (54 mm > 45 mm threshold when AVR indicated); decomposed wording dropped the numeric context, verifier judged the generic reading. Rewrite changed phrasing only, no factual correction."
1,VHD_S8_Q1,verifier_error,"Claim correct per original PDF (>2000 AU men, >1200 AU women; question states same). Corpus prose chunk 8.2.1 has PDF-extraction displacement of numbers vs sex labels; verifier read displaced text linearly and inverted the mapping. Figure_6 in same retrieved top-5 had correct explicit mapping (AVCS >1200 AU female / >2000 AU male) but verifier cited the scrambled prose. Converged; final answer correct."
2,VHD_S8_Q3,verifier_error,"Both claims correct per 8.2.1 (verified against original PDF). Claim A (>3000 AU men 'very likely'): matches the guideline's 'very specific' tier, subset of >2000; 8.2.1 prose was retrieved and cited in all instances, verifier applied rigid single-threshold matching anyway. Claim B (<1600 AU men 'unlikely', run 2 atomic_sec_filter only): near-verbatim guideline; verifier misread 'unlikely' as 'excluded', and retrieval missed the 8.2.1 prose chunk 0163 (top-5: 8.4.1, 8.4.2, Table_4, Figure_6, Table_4; cited chunk Figure_6, which lacks the <1600 sentence), so the verifier inferred from absence. Corrections introduced no errors (final answers state >2000 AU men, correct) but flattened the tiered threshold detail the question asked about. Claim A flagged across all 4 arms in runs 1-2; run 3 clean."
3,VHD_S8_Q9,mixed,"Claim A (MG ≥60 mmHg defines very severe AS): verifier_error. Correct verbatim in Recommendation_Table_4 (IIa B asymptomatic row: 'Very severe AS (mean gradient ≥60 mmHg or Vmax >5.0 m/s)') and in the question. Verifier anchored on 8.2.1 prose, which lacks the threshold; coarse never retrieved Table_4, atomic retrieved Table_4 rows but not evidently the very-severe row (check atomic:0173/0174/0175 content). Unresolved at final iteration in run1 atomic and run2 atomic_gl_filter, but final answers correctly retained ≥60 mmHg. Claim B (severity best represented by maximal Doppler value): true_hallucination, overstatement contradicting 8.2.1's multiparametric grading with mean gradient as most robust parameter; rewrites fixed framing correctly."
4,VHD_S9_Q1,verifier_error,"(1) Minor wording gap: 'generally preferred' is slightly broader than the guideline's tenting-area-specific 'usually required', but the causal content is the guideline's own (9.2.4). (2) Outcome impact is negative: the rewrites replaced a correct SMR-specific statement with the generic 'repair preferred when durable' framing, so this false positive made final answers slightly less accurate for this patient, unlike VHD_S8_Q1 and VHD_S8_Q3, where the corrections left the final answers correct."
...,...,...,...
86,AF_S9_Q10,verifier_error,"9 refuted claim instances in 7 of 12 run-arm instances; all wrong-population phantoms applying the AF antiplatelet-prohibition rule to an ESUS patient without documented AF, contrary to Recommendation_Table_27 (III A) and 9.7; 1 correction-introduced error shipped in run1 as."
87,AF_S10_Q2,cross_source,"4 refuted claim instances in 4 of 12 run-arm instances; all absence-as-refutation of the externally standard five-class first-line antihypertensive list against Rec_Table_32's ACE-I/ARB-only naming; run3 as resisted, 3 finals narrowed."
88,CM_S7-4_Q1,true_hallucination,"7 refuted claim instances in 7 of 12 run-arm instances; both claim types assert sotalol escalation/preference contradicted by the corpus hierarchy (amiodarone IIa C, sotalol limited and conflicting, no recommendation row); correct catches, clean convergence."
89,CVD_S3-3_Q1,true_hallucination,"26 refuted claim instances in 12 of 12 run-arm instances, the largest batch-4 count; all 5 claim types are fabricated or misassigned ethnicity multipliers against 3.3.2; verifier caught all initial errors, but 3 correction loops manufactured new numeric

In [ ]:
display(df)

,uid,claim_id,claim_text,label,n_instances,arms_flagged,runs_flagged,cited_chunk,source_chunk_retrieved,outcome,note
0,VHD_S7_Q2,A,Aortic size alone strongly mandates aortic intervention in the setting of planned valve surgery,verifier_error,2/12,"a, ag",1,atomic:0163->0139 (prose 7.4),yes,converged; final answers kept correct 45 mm content,"Phantom via decontextualized qualitative quibble, confirmed with full refs. 7.4: 'The presence of associated aortic dilatation dictates surgery, irrespective of AR severity' and concomitant aortic surgery should be considered at >=45 mm when AV surgery is indicated; Recommendation_Table_3 gives the same >=45 mm rule (IIa C), Figure_5 note_b repeats it. In the vignette (54 mm with AVR indicated, also above the 50 mm risk-factor tier) aortic size independently mandates intervention, so the claim is corpus-consistent in context. Decomposition dropped the 54 mm vs 45 mm numbers; the verifier refuted the generic reading against its own rank-1 chunk. Within-run inconsistency: identical initial answer unflagged in run1 c and as; runs 2-3 clean."
1,VHD_S8_Q1,A,AS calcium thresholds >1200 AU women / >2000 AU men,verifier_error,1/12,as,3,atomic:0163 (prose 8.2.1),yes (extraction-displaced),converged; final answer has correct thresholds,"Extraction-artifact phantom. Guideline mapping is AVCS >1200 AU women / >2000 AU men, stated explicitly in Figure_6 nodes ('AVCS >1200 AU (female) >2000 AU (male)') and in the question itself. The 8.2.1 prose chunk has PDF-extraction displacement of the number sequence, so a linear read inverts the sex pairing (1200 with men, displaced 2000 with women). The verifier read the scrambled prose linearly and refuted the correct claim; Figure_6 at rank 1 in the same top-5 was bypassed. Root cause split: corpus extraction artifact primary, verifier cross-check failure secondary. Clinical conclusion unaffected under either reading (2500 AU exceeds both). Within-run inconsistency: identical initial answer unflagged in run3 c/a/ag; runs 1-2 clean."
2,VHD_S8_Q3,A,Severe AS very likely when calcium score >3000 AU in men,verifier_error,8/12,4,"1, 2",atomic:0163 / coarse:0103 (prose 8.2.1),yes,"converged; final answers state >2000 AU (correct, tiered detail lost)","Tier-collapse phantom. 8.2.1 is two-tier: >2000 AU men / >1200 AU women indicate severe AS with high sensitivity and specificity (~85%); higher thresholds >3000 men / >1600 women are very specific; unlikely <1600 men / <800 women. The claim paraphrases the very-specific tier as 'very likely' and is corpus-consistent. The verifier quoted only the >2000 lower-tier sentence in all 8 instances and refuted the upper tier within the same chunk: the same within-chunk evidence-selection failure as VHD_S8_Q4 claim B and VHD_S8_Q7 claim A. Asymmetry with VHD_S8_Q1: sex mapping read correctly here, only the tiers collapsed. Unflagged minors: dimensionless index <0.25 absent from pasted 8.2.1/Figure_6 (external standard, not contradicted); initial answers define low flow as SVI <35 against the corpus's inclusive <=35 boundary, never flagged."
3,VHD_S8_Q3,B,Severe AS unlikely when calcium score <1600 AU in men,verifier_error,1/12,as,2,atomic:0190 (Figure_6),no (8.2.1 prose 0163 absent from top-5),converged,"Hedge-misreading phantom plus retrieval miss. 8.2.1 verbatim: 'severe AS becomes unlikely in patients with calcium AV scoring of <1600 AU in men and <800 AU in women'; the claim is near-verbatim, hedge included. The verifier misread 'unlikely' as 'excluded' and refuted an exclusion the claim never asserted, inferring from Figure_6's silence after the governing prose chunk fell out of the top-5."
4,VHD_S8_Q9,A,Very severe AS defined by mean gradient >=60 mmHg,verifier_error,10/12,4,"1, 2, 3",atomic:0163 / coarse:0103 (prose 8.2.1),"no (definition is in Recommendation_Table_4 IIa asymptomatic row; coarse: Table_4 not retrieved; atomic: rows 0173/0175 retrieved, check if very-severe bullet attached)",unresolved at final in r1 atomic

In [ ]:
# th_subtype vocabulary (true_hallucination rows only):
#   numeric_threshold      wrong value, threshold, percentage, or boundary
#   class_level            wrong or fabricated recommendation class/level
#   population_scope       wrong patient group, indication, or definitional scope
#   strength_logic         overstated, inverted, or misordered strength/conditions
#   fabricated_attribution invented number/attribution presented as corpus content
# rule: assign the type of the deviation the verifier engaged

TH_SUBTYPE = {
    # numeric_threshold (27)
    ("VHD_S14_Q1","A"):"numeric_threshold", ("VHD_S6_Q7","A"):"numeric_threshold",
    ("VHD_S7_Q5","A"):"numeric_threshold", ("VHD_S7_Q5","B"):"numeric_threshold",
    ("VHD_S14_Q2","A"):"numeric_threshold", ("VHD_S6_Q1","A"):"numeric_threshold",
    ("VHD_S6_Q5","A"):"numeric_threshold", ("VHD_S8_Q4","A"):"numeric_threshold",
    ("VHD_S8_Q7","B"):"numeric_threshold", ("VHD_S9_Q5","A"):"numeric_threshold",
    ("AF_S3_Q4","A"):"numeric_threshold", ("AF_S3_Q3","A"):"numeric_threshold",
    ("AF_S5_Q3","A"):"numeric_threshold", ("AF_S6_Q4","A"):"numeric_threshold",
    ("AF_S9_Q2","A"):"numeric_threshold", ("AF_S10_Q3","A"):"numeric_threshold",
    ("CM_S7-5_Q1","A"):"numeric_threshold", ("CVD_S6_Q1","A"):"numeric_threshold",
    ("CVD_S4-7_Q1","B"):"numeric_threshold", ("CVD_S4-7_Q1","C"):"numeric_threshold",
    ("CVD_S3-3_Q1","A"):"numeric_threshold", ("CVD_S3-3_Q1","D"):"numeric_threshold",
    ("CVD_S3-3_Q1","E"):"numeric_threshold", ("CVD_S4-5_Q1","A"):"numeric_threshold",
    ("CVD_S4-5_Q1","B"):"numeric_threshold",
    # class_level (8)
    ("VHD_S6_Q4","A"):"class_level", ("VHD_S6_Q4","B"):"class_level",
    ("VHD_S7_Q7","B"):"class_level", ("VHD_S9_Q5","C"):"class_level",
    ("AF_S7_Q1","A"):"class_level", ("CM_S7-1_Q2","C"):"class_level",
    # population_scope (6)
    ("VHD_S5_Q4","B"):"population_scope", ("AF_S6_Q7","A"):"population_scope",
    ("AF_S7_Q6","A"):"population_scope", ("AF_S9_Q2","B"):"population_scope",
    ("CM_S3_Q1","A"):"population_scope", ("CM_S8_Q1","A"):"population_scope",
    # strength_logic (15)
    ("VHD_S8_Q9","B"):"strength_logic", ("VHD_S8_Q8","B"):"strength_logic",
    ("VHD_S5_Q1","A"):"strength_logic", ("VHD_S5_Q1","C"):"strength_logic",
    ("VHD_S14_Q8","A"):"strength_logic", ("AF_S3_Q2","A"):"strength_logic",
    ("AF_S9_Q11","E"):"strength_logic", ("CM_S12_Q1","A"):"strength_logic",
    ("CM_S7-1_Q2","A"):"strength_logic", ("CM_S7-1_Q2","B"):"strength_logic",
    ("CM_S7-1_Q3","A"):"strength_logic", ("CM_S7-1_Q3","B"):"strength_logic",
    ("CM_S7-1_Q3","C"):"strength_logic", ("CM_S7-4_Q1","A"):"strength_logic",
    ("CM_S7-4_Q1","B"):"strength_logic",
    # fabricated_attribution (6)
    ("AF_S8_Q1","A"):"fabricated_attribution", ("AF_S8_Q1","B"):"fabricated_attribution",
    ("AF_S10_Q3","B"):"fabricated_attribution", ("CVD_S3-3_Q1","B"):"fabricated_attribution",
    ("CVD_S3-3_Q1","C"):"fabricated_attribution", ("CVD_S4-5_Q1","C"):"fabricated_attribution",
}
for (u, cid), st in TH_SUBTYPE.items():
    upsert_claim(u, cid, th_subtype=st)

# sanity check against the real label column
chk = pd.read_csv(CLAIMS_CSV, dtype=str).fillna("")
th_chk = chk[chk["label"] == "true_hallucination"]
print(f"TH rows: {len(th_chk)} (expect 59), with subtype: {(th_chk['th_subtype'] != '').sum()}")
print(th_chk["th_subtype"].value_counts())

saved VHD_S14_Q1 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S6_Q7 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S7_Q5 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S7_Q5 B -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S14_Q2 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S6_Q1 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S6_Q5 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S8_Q4 A -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S8_Q7 B -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
saved VHD_S9_Q5 A -> /content/drive/MyDrive/medical_qa_halluci

In [ ]:
# ============ CELL A4 : build TH confirmation workbook for the expert ========
# Reads audit_claims.csv, keeps the true_hallucination rows, joins question
# text + guideline_ref from the dataset, pulls corpus excerpts for the cited
# chunks, and writes one bilingual (DE instructions / EN content) workbook.
from config import INDEX_DIR
import re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

CLAIMS_CSV = os.path.join(AUDIT_DIR, "audit_claims.csv")
OUT_XLSX   = os.path.join(AUDIT_DIR, "th_confirmation_expert.xlsx")
assert os.path.exists(CLAIMS_CSV), f"not found: {CLAIMS_CSV}"

claims = pd.read_csv(CLAIMS_CSV, dtype=str).fillna("")
th = claims[claims["label"] == "true_hallucination"].copy()
assert len(th) == 62, f"expected 62 TH claim rows, found {len(th)}"

META = {}
for name in ("atomic", "coarse"):
    with open(os.path.join(INDEX_DIR, f"{name}_meta.jsonl"), encoding="utf-8") as f:
        META[name] = [json.loads(l) for l in f if l.strip()]
print("meta loaded:", {n: len(m) for n, m in META.items()})

def _parse_cited(cited):
    """'atomic:0163->0139 (prose 7.4) / coarse:0103' -> [('atomic',163), ...]"""
    out, cur = [], None
    for tok in re.finditer(r"(atomic|coarse):(\d+)|->\s*(\d+)", cited or ""):
        if tok.group(1):
            cur = tok.group(1); out.append((cur, int(tok.group(2))))
        elif tok.group(3) and cur:
            out.append((cur, int(tok.group(3))))
    seen, res = set(), []
    for x in out:
        if x not in seen:
            seen.add(x); res.append(x)
    return res

def _excerpt(cited, max_len=1200):
    parts = []
    for name, pos in _parse_cited(cited):
        if pos >= len(META[name]):
            continue
        m = META[name][pos]
        loc = (m.get("section") or m.get("table_id") or m.get("figure_id") or "?")
        t = m.get("text", "").strip()
        if len(t) > max_len:
            t = t[:max_len].rstrip() + " ..."
        parts.append(f"[{name}:{pos:04d} | {loc}]\n{t}")
    return "\n\n".join(parts)

def _qtext(uid):
    r = DS_BY_UID.get(uid, {})
    return r.get("final_question_text") or r.get("question_text") or ""

rows, missing_excerpt = [], []
for _, r in th.iterrows():
    ex = _excerpt(r["cited_chunk"])
    if not ex:
        missing_excerpt.append(f"{r['uid']} {r['claim_id']}")
    rows.append({
        "uid": r["uid"], "claim_id": r["claim_id"],
        "group": UID2GROUP.get(r["uid"], "?"),
        "guideline": r["uid"].split("_")[0],
        "question": _qtext(r["uid"]),
        "claim": r["claim_text"],
        "subtype": r["th_subtype"],
        "flagged": f"{r['n_instances']}; arms {r['arms_flagged']}; runs {r['runs_flagged']}",
        "ds_refs": _refs_str(r["uid"]),
        "excerpt": ex,
        "note": r["note"],
    })
print(f"{len(rows)} TH rows prepared; rows WITHOUT auto excerpt: "
      f"{len(missing_excerpt)} -> {missing_excerpt}")

# ---- workbook ---------------------------------------------------------------
wb = Workbook()

ws0 = wb.active
ws0.title = "Anleitung"
anleitung = [
    "Bestaetigung der Halluzinations-Befunde (Masterarbeit)",
    "",
    "Hintergrund:",
    "Ein Sprachmodell hat 160 kardiolologische Fragen beantwortet (ESC-Leitlinien:",
    "VHD 2025, AF 2024, Kardiomyopathien 2023, CVD-Praevention 2021). Eine",
    "Pruef-Pipeline hat einzelne Aussagen der Antworten gegen die Leitlinientexte",
    "geprueft. Diese Datei enthaelt die 62 Aussagen (aus 40 Fragen), die meine",
    "manuelle Pruefung als echte Halluzination eingestuft hat: die Aussage",
    "widerspricht dem Inhalt der ESC-Leitlinie.",
    "",
    "Ihre Aufgabe pro Zeile:",
    "1. Behauptung (Spalte F) gegen den Leitlinien-Auszug (Spalte I) pruefen.",
    "2. Spalte K ausfuellen: J = trifft zu (die Aussage widerspricht der Leitlinie),",
    "   N = trifft nicht zu (die Aussage ist fachlich vertretbar), unsicher.",
    "3. Spalte L: optionaler Kommentar, besonders bei N oder unsicher.",
    "",
    "Hinweise:",
    "- Alle Inhalte sind auf Englisch (Original-Leitlinientext und Modellaussagen).",
    "- Spalte H nennt die Referenzen, aus denen die Frage erstellt wurde;",
    "  Spalte I zeigt die Textstellen, die die Pipeline tatsaechlich heranzog.",
    "- Spalte J ist meine Audit-Begruendung (Arbeitsnotiz).",
    "",
    "EN summary: 62 claims from 40 questions, each judged a true hallucination",
    "against the ESC guideline text. Per row: check the claim against the excerpt,",
    "mark J (agree) / N (disagree) / unsicher in column K, comment in L if useful.",
]
for i, line in enumerate(anleitung, 1):
    ws0.cell(row=i, column=1, value=line)
ws0["A1"].font = Font(bold=True, size=13)
ws0.column_dimensions["A"].width = 100

ws = wb.create_sheet("Faelle")
HEADERS = ["Kennung\nuid", "Claim", "Gruppe", "Leitlinie",
           "Frage (Englisch)", "Behauptung (Englisch)", "Typ",
           "Markiert\ninstances/arms/runs", "Referenzen (Datensatz)",
           "Leitlinien-Auszug (Korpus)", "Audit-Bewertung (Englisch)",
           "Zutreffend?\n(J/N/unsicher)", "Kommentar"]
KEYS = ["uid", "claim_id", "group", "guideline", "question", "claim",
        "subtype", "flagged", "ds_refs", "excerpt", "note"]
hfill = PatternFill("solid", fgColor="DCE6F1")
for c, h in enumerate(HEADERS, 1):
    cell = ws.cell(row=1, column=c, value=h)
    cell.font = Font(bold=True); cell.fill = hfill
    cell.alignment = Alignment(wrapText=True, vertical="center")
for i, row in enumerate(rows, 2):
    for c, k in enumerate(KEYS, 1):
        cell = ws.cell(row=i, column=c, value=row[k])
        cell.alignment = Alignment(wrapText=True, vertical="top")
widths = [16, 6, 11, 9, 55, 45, 18, 16, 22, 60, 50, 12, 25]
for c, w in enumerate(widths, 1):
    ws.column_dimensions[get_column_letter(c)].width = w
ws.freeze_panes = "C2"

dv = DataValidation(type="list", formula1='"J,N,unsicher"', allow_blank=True,
                    showDropDown=False)
ws.add_data_validation(dv)
dv.add(f"L2:L{len(rows)+1}")

ws2 = wb.create_sheet("Legende")
ws2.append(["Typ", "Bedeutung (EN)", "Bedeutung (DE)"])
for c in range(1, 4):
    ws2.cell(row=1, column=c).font = Font(bold=True)
legend = [
    ("numeric_threshold", "wrong number, threshold or range",
     "falsche Zahl, Schwelle oder Bereich"),
    ("strength_logic", "overstated strength or inverted logic of a statement",
     "ueberzogene Aussagestaerke oder umgekehrte Logik"),
    ("class_level", "wrong recommendation class or level of evidence",
     "falsche Empfehlungsklasse oder Evidenzlevel"),
    ("population_scope", "claim applied to the wrong patient population",
     "Aussage auf falsche Patientenpopulation bezogen"),
    ("fabricated_attribution", "content fabricated and attributed to the guideline",
     "Inhalt erfunden und der Leitlinie zugeschrieben"),
    ("", "", ""),
    ("J", "agree: the claim contradicts the guideline",
     "trifft zu: die Aussage widerspricht der Leitlinie"),
    ("N", "disagree: the claim is defensible",
     "trifft nicht zu: die Aussage ist fachlich vertretbar"),
    ("unsicher", "uncertain", "unsicher"),
]
for row in legend:
    ws2.append(list(row))
ws2.column_dimensions["A"].width = 24
ws2.column_dimensions["B"].width = 55
ws2.column_dimensions["C"].width = 55

wb.save(OUT_XLSX)
print(f"\nSAVED -> {OUT_XLSX}  ({len(rows)} rows, 40 uids expected: "
      f"{len({r['uid'] for r in rows})} uids)")

meta loaded: {'atomic': 1035, 'coarse': 525}
62 TH rows prepared; rows WITHOUT auto excerpt: 0 -> []

SAVED -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/th_confirmation_expert.xlsx  (62 rows, 40 uids expected: 40 uids)


In [ ]:
# ============ CELL R1 : true_hallucination review workbook ============
import pandas as pd, os

claims = pd.read_csv(CLAIMS_CSV, dtype=str).fillna("")
th = claims[claims["label"] == "true_hallucination"].copy()
th_uids = list(dict.fromkeys(th["uid"]))          # preserves audit order
print(f"{len(th_uids)} uids with >=1 true_hallucination row, {len(th)} TH claim rows")

def _get(d, *cands):
    for c in cands:
        v = d.get(c)
        if v:
            return v
    return ""

def _refuted_texts(arm):
    # schema: history[i]["verified"] = all claims, keep verdict == "refuted"
    out = []
    for it, h in enumerate(arm.get("history", []), 1):
        if not isinstance(h, dict):
            continue
        for c in h.get("verified", []) or []:
            if isinstance(c, dict) and c.get("verdict") == "refuted":
                claim = _get(c, "claim", "claim_text")
                why = _get(c, "rationale", "why", "reason")
                out.append(f"[it{it}] {claim} -- WHY: {why}")
    return "\n".join(out)

def _clip(x, n=32000):                            # excel cell limit is 32767
    return x[:n] if isinstance(x, str) and len(x) > n else x

ov_rows, inst_rows = [], []
for u in th_uids:
    g = UID2GROUP[u]
    uc = th[th["uid"] == u]
    ov_rows.append({
        "uid": u, "group": g, "guideline_ref": _refs_str(u),
        "th_claim_types": len(uc),
        "th_instances": sum(int(x.split("/")[0]) for x in uc["n_instances"]),
        "claims": "\n".join(f"{r.claim_id}. {r.claim_text}  [{r.n_instances}; "
                            f"arms {r.arms_flagged}; runs {r.runs_flagged}]"
                            for r in uc.itertuples()),
        "audit_notes": "\n".join(f"{r.claim_id}: {r.note}"
                                 for r in uc.itertuples()),
    })
    for ri, d in enumerate(RUNS[g], 1):
        r = next(x for x in d["results"] if x["uid"] == u)
        init = _get(r, "init_answer", "initial_answer", "answer_initial", "initial")
        for a in ARM_ORDER:
            arm = r["arms"][a]
            inst_rows.append({
                "uid": u, "run": ri, "arm": ARM_CODES[a],
                "ref_init": arm["ref_init"],
                "final_refuted": arm["final_refuted"],
                "refuted_claims": _refuted_texts(arm),
                "initial_answer": init,
                "final_answer": _get(arm, "final_answer", "answer_final", "final"),
            })

ov = pd.DataFrame(ov_rows).map(_clip)
inst = pd.DataFrame(inst_rows).map(_clip)

# claims sheet: TH rows only, label dropped (implied), th_subtype added
if "th_subtype" not in th.columns:
    th["th_subtype"] = ""
sub = th[["uid", "claim_id", "claim_text", "th_subtype", "n_instances",
          "arms_flagged", "runs_flagged", "cited_chunk",
          "source_chunk_retrieved", "outcome", "note"]]

OUT = os.path.join(AUDIT_DIR, "th_review.xlsx")
with pd.ExcelWriter(OUT, engine="openpyxl") as w:
    ov.to_excel(w, sheet_name="overview", index=False)
    sub.to_excel(w, sheet_name="claims", index=False)
    inst.to_excel(w, sheet_name="instances", index=False)

from openpyxl import load_workbook
from openpyxl.styles import Alignment
wb = load_workbook(OUT)
WIDE = {"claims", "audit_notes", "refuted_claims", "initial_answer",
        "final_answer", "claim_text", "note", "outcome"}
for ws in wb.worksheets:
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = \
            90 if col[0].value in WIDE else 16
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
wb.save(OUT)
print("saved ->", OUT)
print(f"claims sheet rows: {len(sub)} (expect 59)")
print(inst[["refuted_claims", "initial_answer"]].replace("", pd.NA).notna().sum())

39 uids with >=1 true_hallucination row, 59 TH claim rows
saved -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/th_review.xlsx
claims sheet rows: 59 (expect 59)
refuted_claims    279
initial_answer    468
dtype: int64


In [ ]:
# CELL S1: class/level mentions in final answers vs pipeline catches
import json, re, glob
import pandas as pd

RESULT_FILES = (
    sorted(glob.glob("/content/drive/MyDrive/medical_qa_hallucination/experiments/results/final/results_batch*_run*.json"))
    + sorted(glob.glob("/content/drive/MyDrive/medical_qa_hallucination/experiments/results/results_reviewed20_run*.json"))
)
print(len(RESULT_FILES), "result files found:")  # expect 15

ARM_KEYS = ["coarse", "atomic", "atomic_gl_filter", "atomic_sec_filter"]
CLASS_RE = re.compile(r"\b[Cc]lass\s+(I{1,3}[ab]?|IV)\b|\b[Ll]evel\s+[ABC]\b")

def _final_text(d):
    v = d.get("final_answer")
    return v if isinstance(v, str) else None

def _refuted_texts(d):
    # only claims with verdict 'refuted', across all iterations in history
    out = []
    for it in d.get("history", []) or []:
        if not isinstance(it, dict):
            continue
        for c in it.get("verified", []) or []:
            if isinstance(c, dict) and c.get("verdict") == "refuted":
                t = c.get("claim")
                if isinstance(t, str):
                    out.append(t)
    return out

def iter_instances(obj):
    recs = obj if isinstance(obj, list) else obj.get("results", obj.get("data", obj.get("records", [])))
    for rec in recs:
        uid = rec.get("uid")
        for arm in ARM_KEYS:
            d = rec.get("arms", {}).get(arm)
            if isinstance(d, dict):
                yield uid, arm, d

rows, mention_rows, missing = [], [], 0
for fp in RESULT_FILES:
    run = re.search(r"run(\d+)", fp)
    run = run.group(1) if run else fp
    with open(fp) as f:
        data = json.load(f)
    for uid, arm, d in iter_instances(data):
        final = _final_text(d)
        if not final:
            missing += 1
            continue
        mentions = [(m.group(0), final[max(0, m.start()-70):m.end()+70].replace("\n", " "))
                    for m in CLASS_RE.finditer(final)]
        class_claims_refuted = [t for t in _refuted_texts(d) if CLASS_RE.search(t)]
        rows.append({"uid": uid, "run": run, "arm": arm,
                     "n_mentions": len(mentions),
                     "class_claim_refuted": len(class_claims_refuted)})
        for txt, ctx in mentions:
            mention_rows.append({"uid": uid, "run": run, "arm": arm,
                                 "mention": txt, "context": ctx,
                                 "class_claim_refuted_in_instance": bool(class_claims_refuted)})

df = pd.DataFrame(rows)
md = pd.DataFrame(mention_rows)

print(f"\ninstances parsed: {len(df)} | finals missing/unparsed: {missing}")
if len(df):
    print(f"final answers with >=1 class/level mention: {(df.n_mentions > 0).sum()} of {len(df)}")
    print(f"total mentions in finals: {df.n_mentions.sum()}")
    print(f"instances where >=1 refuted claim contained a class/level label: {(df.class_claim_refuted > 0).sum()}")
    print(f"mentions in finals of instances with NO class-claim refutation: "
          f"{df[(df.n_mentions > 0) & (df.class_claim_refuted == 0)].n_mentions.sum()}")

    print("\nmentions per arm:")
    print(df.groupby("arm").agg(finals=("uid", "size"),
                                finals_with_mentions=("n_mentions", lambda s: (s > 0).sum()),
                                mentions=("n_mentions", "sum"),
                                caught=("class_claim_refuted", lambda s: (s > 0).sum())))

    print("\nmentions per run:")
    print(df.groupby("run").agg(mentions=("n_mentions", "sum"),
                                caught=("class_claim_refuted", lambda s: (s > 0).sum())))

    surv = md[~md.class_claim_refuted_in_instance].drop_duplicates(subset=["uid", "run", "arm", "context"])
    print(f"\nunique surviving mention contexts to eyeball: {len(surv)}")
    with pd.option_context("display.max_colwidth", 200, "display.max_rows", None):
        display(surv[["uid", "run", "arm", "mention", "context"]])

    # surv.to_csv("class_mentions_review.csv", index=False, encoding="utf-8")

15 result files found:

instances parsed: 1920 | finals missing/unparsed: 0
final answers with >=1 class/level mention: 505 of 1920
total mentions in finals: 666
instances where >=1 refuted claim contained a class/level label: 46
mentions in finals of instances with NO class-claim refutation: 578

mentions per arm:
                   finals  finals_with_mentions  mentions  caught
arm                                                              
atomic                480                   129       167      12
atomic_gl_filter      480                   129       169       9
atomic_sec_filter     480                   124       163      11
coarse                480                   123       167      14

mentions per run:
     mentions  caught
run                  
1         216      11
2         243       8
3         207      27

unique surviving mention contexts to eyeball: 578


,uid,run,arm,mention,context
0,VHD_S5_Q2,1,coarse,Class IIa,"ormal range on repeated measurements without another explanation (ESC Class IIa), because this identifies a higher-risk stage before overt LV dysfunc"
1,VHD_S5_Q2,1,atomic,Class IIa,"ormal range on repeated measurements without another explanation (ESC Class IIa), because this identifies a higher-risk stage before overt LV dysfunc"
2,VHD_S5_Q2,1,atomic_gl_filter,Class IIa,"ormal range on repeated measurements without another explanation (ESC Class IIa), because this identifies a higher-risk stage before overt LV dysfunc"
3,VHD_S5_Q2,1,atomic_sec_filter,Class IIa,"ormal range on repeated measurements without another explanation (ESC Class IIa), because this identifies a higher-risk stage before overt LV dysfunc"
4,VHD_S10_Q1,1,coarse,Class I,the main indication for intervention. ESC guidance supports PMC as a Class I indication in symptomatic patients with severe rheumatic mitral steno
5,VHD_S10_Q1,1,atomic,Class I,the main indication for intervention. ESC guidance supports PMC as a Class I indication in symptomatic patients with severe rheumatic mitral steno
6,VHD_S10_Q1,1,atomic_gl_filter,Class I,the main indication for intervention. ESC guidance supports PMC as a Class I indication in symptomatic patients with severe rheumatic mitral steno
7,VHD_S10_Q1,1,atomic_sec_filter,Class I,the main indication for intervention. ESC guidance supports PMC as a Class I indication in symptomatic patients with severe rheumatic mitral steno
8,VHD_S14_Q1,1,atomic,Class I,"in critically ill patients without a contraindication to surgery** (**Class I**), and her **pronounced heart-failure symptoms with a markedly eleva"
9,VHD_S14_Q1,1,atomic_gl_filter,Class I,"in critically ill patients without a contraindication to surgery** (**Class I**), and her **pronounced heart-failure symptoms with a markedly eleva"


In [ ]:
# CELL S1b: final-iteration verdict split for class/level mentions
rows2 = []
for fp in RESULT_FILES:
    run = re.search(r"run(\d+)", fp).group(1)
    with open(fp) as f:
        data = json.load(f)
    for uid, arm, d in iter_instances(data):
        final = d.get("final_answer")
        if not isinstance(final, str):
            continue
        n_ment = len(CLASS_RE.findall(final))
        hist = d.get("history") or []
        last = hist[-1] if hist and isinstance(hist[-1], dict) else {}
        vc = [c for c in last.get("verified", []) or []
              if isinstance(c, dict) and CLASS_RE.search(str(c.get("claim", "")))]
        rows2.append({"uid": uid, "run": run, "arm": arm, "n_mentions": n_ment,
                      "cc_supported": sum(1 for c in vc if c.get("verdict") == "supported"),
                      "cc_refuted": sum(1 for c in vc if c.get("verdict") == "refuted"),
                      "cc_unverifiable": sum(1 for c in vc if c.get("verdict") == "unverifiable")})

d2 = pd.DataFrame(rows2)
w = d2[d2.n_mentions > 0]

print(f"finals with >=1 mention: {len(w)}  ({len(w)/len(d2)*100:.1f}% of {len(d2)})")
print(f"  (1) class claim SUPPORTED at final iteration (checked, passed): {(w.cc_supported > 0).sum()}")
print(f"  (2) class claim REFUTED at final iteration (shipped unresolved): {(w.cc_refuted > 0).sum()}")
print(f"  (3) no class-bearing claim in final verification (never isolated, unchecked): "
      f"{((w.cc_supported == 0) & (w.cc_refuted == 0) & (w.cc_unverifiable == 0)).sum()}")
print(f"  unverifiable class claims at final iteration: {(w.cc_unverifiable > 0).sum()}")

print("\nper arm:")
print(w.groupby("arm").agg(finals=("uid", "size"),
                           supported=("cc_supported", lambda s: (s > 0).sum()),
                           shipped_refuted=("cc_refuted", lambda s: (s > 0).sum()),
                           unchecked=("cc_supported", lambda s: 0)).drop(columns="unchecked"))

# unresolved cases are the interesting failures, list them
bad = w[w.cc_refuted > 0]
print(f"\nfinals that shipped a refuted class claim: {len(bad)}")
with pd.option_context("display.max_rows", None):
    display(bad[["uid", "run", "arm", "n_mentions", "cc_refuted"]])

finals with >=1 mention: 505  (26.3% of 1920)
  (1) class claim SUPPORTED at final iteration (checked, passed): 396
  (2) class claim REFUTED at final iteration (shipped unresolved): 8
  (3) no class-bearing claim in final verification (never isolated, unchecked): 0
  unverifiable class claims at final iteration: 174

per arm:
                   finals  supported  shipped_refuted
arm                                                  
atomic                129         98                3
atomic_gl_filter      129        101                2
atomic_sec_filter     124         98                0
coarse                123         99                3

finals that shipped a refuted class claim: 8


,uid,run,arm,n_mentions,cc_refuted
104,AF_S9_Q11,1,coarse,2,1
105,AF_S9_Q11,1,atomic,5,2
106,AF_S9_Q11,1,atomic_gl_filter,3,1
244,AF_S9_Q11,2,coarse,4,1
386,AF_S9_Q11,3,atomic_gl_filter,3,1
564,VHD_S6_Q4,2,coarse,2,1
1053,AF_S5_Q1,2,atomic,2,1
1437,VHD_S9_Q5,2,atomic,2,1


In [ ]:
# CELL S1c: dump the shipped refuted class claims
for fp in RESULT_FILES:
    run = re.search(r"run(\d+)", fp).group(1)
    with open(fp) as f:
        data = json.load(f)
    for uid, arm, d in iter_instances(data):
        hist = d.get("history") or []
        if not hist:
            continue
        last = hist[-1]
        for c in last.get("verified", []) or []:
            if isinstance(c, dict) and c.get("verdict") == "refuted" and CLASS_RE.search(str(c.get("claim", ""))):
                print(f"{uid} | run {run} | {arm}")
                print(f"  CLAIM : {c.get('claim')}")
                print(f"  WHY   : {c.get('rationale')}")
                print(f"  CHUNK : {c.get('chunk_id')}")
                print()

AF_S9_Q11 | run 1 | coarse
  CLAIM : Both prophylactic amiodarone and peri-operative beta-blocker treatment for prevention of post-operative AF carry a Class IIb recommendation.
  WHY   : Peri-operative amiodarone therapy has a Class I recommendation, and routine use of beta-blockers is Class III, not Class IIb, for prevention of post-operative AF.
  CHUNK : coarse:0265:AF:table_whole:Recommendation_Table_26

AF_S9_Q11 | run 1 | atomic
  CLAIM : Beta-blocker treatment to prevent post-operative AF after cardiac surgery is a Class I recommendation.
  WHY   : Routine use of beta-blockers to prevent post-operative AF is explicitly not recommended (Class III).
  CHUNK : atomic:0474:AF:recommendation:Recommendation_Table_26

AF_S9_Q11 | run 1 | atomic
  CLAIM : Peri-operative amiodarone to reduce post-operative AF is a Class IIb recommendation.
  WHY   : Peri-operative amiodarone therapy to prevent post-operative AF is a Class I recommendation, not Class IIb.
  CHUNK : atomic:0471:AF:recomme

In [ ]:
# CELL S1d: adjudication of the 8 shipped-refuted finals
# (labels from audit_claims.csv ground truth, expert-reviewed)
# 7 verifier-correct residual errors (correction budget exhausted),
# 1 phantom refutation: the shipped claim was corpus-correct
SHIPPED_ADJ = {
    ("AF_S9_Q11", "1", "coarse"):           "verifier_correct",
    ("AF_S9_Q11", "1", "atomic"):           "verifier_correct",
    ("AF_S9_Q11", "1", "atomic_gl_filter"): "verifier_correct",
    ("AF_S9_Q11", "2", "coarse"):           "verifier_correct",
    ("AF_S9_Q11", "3", "atomic_gl_filter"): "verifier_correct",
    ("VHD_S6_Q4",  "2", "coarse"):          "verifier_correct",
    ("AF_S5_Q1",   "2", "atomic"):          "phantom_refutation_correct_claim",
    ("VHD_S9_Q5",  "2", "atomic"):          "verifier_correct",
}
bad = bad.copy()
bad["adjudication"] = [SHIPPED_ADJ.get((r.uid, r.run, r.arm), "?")
                       for r in bad.itertuples()]
assert (bad["adjudication"] != "?").all(), "unadjudicated shipped case"
print(bad["adjudication"].value_counts().to_string())
display(bad[["uid", "run", "arm", "n_mentions", "cc_refuted", "adjudication"]])

adjudication
verifier_correct                    7
phantom_refutation_correct_claim    1


,uid,run,arm,n_mentions,cc_refuted,adjudication
104,AF_S9_Q11,1,coarse,2,1,verifier_correct
105,AF_S9_Q11,1,atomic,5,2,verifier_correct
106,AF_S9_Q11,1,atomic_gl_filter,3,1,verifier_correct
244,AF_S9_Q11,2,coarse,4,1,verifier_correct
386,AF_S9_Q11,3,atomic_gl_filter,3,1,verifier_correct
564,VHD_S6_Q4,2,coarse,2,1,verifier_correct
1053,AF_S5_Q1,2,atomic,2,1,phantom_refutation_correct_claim
1437,VHD_S9_Q5,2,atomic,2,1,verifier_correct


In [ ]:
upsert_claim("VHD_S5_Q4", "B", source_chunk_retrieved="yes")

saved VHD_S5_Q4 B -> /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
